<a href="https://colab.research.google.com/github/MatiasMoreno707/lab07-lp/blob/dev/lab07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

### a. Calcule el information value (IV), tanto para el grupo de variable numéricas como categóricas y excluya las que tenga un poder predictivo débil o menor. Además, separe la variable de clasificación del resto de variables para luego obtener los datos de entrenamiento y prueba, tomando de este último el 25% de datos.

In [13]:
def cargar_dataset(url):
  """
  Retrive data from a csv into a dataframe.
  input:
    -url: URL of the csv file
  output:
    -dataframe
  """
  columnas = [
        "ID",
        "Clump_Thickness",
        "Uniformity_of_Cell_Size",
        "Uniformity_of_Cell_Shape",
        "Marginal_Adhesion",
        "Single_Epithelial_Cell_Size",
        "Bare_Nuclei",
        "Bland_Chromatin",
        "Normal_Nucleoli",
        "Mitoses",
        "Class"
    ]
  return pd.read_csv(url, names=columnas)


In [15]:
def clean_data(df):
  """
  Clean data from a dataframe.
  input:
    -dataframe
  output:
    -dataframe
  """
  df = df.replace('?', np.nan)
  for col in df.columns[1:-1]:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col].fillna(df[col].mean(), inplace=True)
  return df

In [19]:
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"
df = cargar_dataset(URL)
df.head()

,ID,Clump_Thickness,Uniformity_of_Cell_Size,Uniformity_of_Cell_Shape,Marginal_Adhesion,Single_Epithelial_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1,3,1,1,2
1,1002945,5,4,4,5,7,10,3,2,1,2
2,1015425,3,1,1,1,2,2,3,1,1,2
3,1016277,6,8,8,1,3,4,3,7,1,2
4,1017023,4,1,1,3,2,1,3,1,1,2


In [21]:
df = clean_data(df)
df.head()

<ipython-input-15-d3220dbd8f81>:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)


,ID,Clump_Thickness,Uniformity_of_Cell_Size,Uniformity_of_Cell_Shape,Marginal_Adhesion,Single_Epithelial_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,2
1,1002945,5,4,4,5,7,10.0,3,2,1,2
2,1015425,3,1,1,1,2,2.0,3,1,1,2
3,1016277,6,8,8,1,3,4.0,3,7,1,2
4,1017023,4,1,1,3,2,1.0,3,1,1,2


In [22]:
df.value_counts(["Bare_Nuclei"])

,count
Bare_Nuclei,
1.000000,402
10.000000,132
2.000000,30
5.000000,30
3.000000,28
8.000000,21
4.000000,19
3.544656,16
9.000000,9


In [23]:
df["Class"] = df["Class"].map({2: 0, 4: 1})  # 2: benigno, 4: maligno

In [25]:
df = df.drop("ID", axis=1)

In [26]:
def calc_iv(df, feature, target, bins=10):
  df = df[[feature, target]].copy()
  if df[feature].dtype != "object" and len(df[feature].unique()) > bins:
    df[feature] = pd.qcut(df[feature], q=bins, duplicates='drop')

  iv_df = df.groupby(feature)[target].agg(['count', 'sum'])
  iv_df.columns = ['total', 'bad']
  iv_df['good'] = iv_df['total'] - iv_df['bad']
  iv_df['dist_good'] = iv_df['good'] / iv_df['good'].sum()
  iv_df['dist_bad'] = iv_df['bad'] / iv_df['bad'].sum()
  iv_df['woe'] = np.log((iv_df['dist_good'] + 1e-6) / (iv_df['dist_bad'] + 1e-6))
  iv_df['iv'] = (iv_df['dist_good'] - iv_df['dist_bad']) * iv_df['woe']
  return iv_df['iv'].sum()


In [29]:
iv_dict = {}
features = [col for col in df.columns if col not in ["Class"]]
for col in features:
  iv = calc_iv(df, col, "Class")
  iv_dict[col] = iv

<ipython-input-26-174d2512d051>:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  iv_df = df.groupby(feature)[target].agg(['count', 'sum'])


In [30]:
iv_df = pd.DataFrame.from_dict(iv_dict, orient='index', columns=['IV']).sort_values(by='IV', ascending=False)
print("Information Value por variable:")
print(iv_df)

Information Value por variable:
                                   IV
Uniformity_of_Cell_Size      9.464010
Uniformity_of_Cell_Shape     8.446808
Clump_Thickness              6.538207
Bland_Chromatin              6.218398
Normal_Nucleoli              5.592932
Marginal_Adhesion            4.657950
Bare_Nuclei                  4.550327
Single_Epithelial_Cell_Size  4.015118
Mitoses                      2.316908


In [31]:
filtered_features = iv_df[iv_df['IV'] >= 0.02].index.tolist()
X = df[filtered_features]
y = df["Class"]

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


### b. Genere el modelo de regresión logística y evalúe la exclusión de variables mediante la significancia de los coeficientes. Además, calcule las métricas de clasificación que se implementaron en la parte práctica e interprete sus resultados más importantes.

In [33]:
import statsmodels.api as sm

X_train_sm = sm.add_constant(X_train)
logit_model = sm.Logit(y_train, X_train_sm).fit()
print(logit_model.summary())


Optimization terminated successfully.
         Current function value: 0.084820
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                  Class   No. Observations:                  524
Model:                          Logit   Df Residuals:                      514
Method:                           MLE   Df Model:                            9
Date:                Thu, 01 May 2025   Pseudo R-squ.:                  0.8691
Time:                        00:29:44   Log-Likelihood:                -44.446
converged:                       True   LL-Null:                       -339.63
Covariance Type:            nonrobust   LLR p-value:                2.439e-121
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
const                          -9.5727      1.184     -8.086      0.000   

In [38]:
# Obtener p-values
p_values = logit_model.pvalues
significant_vars = p_values[p_values < 0.05].index.tolist()

# Quitar 'const' si aparece
if 'const' in significant_vars:
    significant_vars.remove('const')

# Redefinir X con solo las variables significativas
X_train_sig = X_train[significant_vars]
X_test_sig = X_test[significant_vars]


In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Entrenar modelo
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predicción
y_pred = model.predict(X_test)

print("Matriz de confusión:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred))


Matriz de confusión:
 [[117   1]
 [  6  51]]

Reporte de clasificación:
               precision    recall  f1-score   support

           0       0.95      0.99      0.97       118
           1       0.98      0.89      0.94        57

    accuracy                           0.96       175
   macro avg       0.97      0.94      0.95       175
weighted avg       0.96      0.96      0.96       175



### c. Genere el modelo SVM, calcule sus métricas de clasificación y compárelas con las del modelo de regresión logística para ver si hubo o no mejoras.

In [40]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

In [41]:
# Modelo SVM
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_sig, y_train)

# Predicciones
y_pred_svm = svm_model.predict(X_test_sig)

In [43]:
# Métricas SVM
print("🔹 Matriz de confusión (SVM):\n", confusion_matrix(y_test, y_pred_svm))
print("\n🔹 Reporte de clasificación (SVM):\n", classification_report(y_test, y_pred_svm))


🔹 Matriz de confusión (SVM):
 [[115   3]
 [  5  52]]

🔹 Reporte de clasificación (SVM):
               precision    recall  f1-score   support

           0       0.96      0.97      0.97       118
           1       0.95      0.91      0.93        57

    accuracy                           0.95       175
   macro avg       0.95      0.94      0.95       175
weighted avg       0.95      0.95      0.95       175

